# Entrenamientos con pares (#20)
Entrena los modelos de la tabla de `docs/METODO.md` con los pares validados y sube `best.pt`, las curvas y `resultado.json` a `gs://vmseg-imagenes/modelos/<NOMBRE>/`.
Para GPU: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU.

Para cada modelo: pegar su bloque en la celda de configuración y correr esa celda y las tres siguientes (las dos primeras, una sola vez por sesión). Abrir Colab con una sola cuenta de Google (la de UdeSA): con varias, `auth.authenticate_user()` falla.

| # | NOMBRE | ARQUITECTURA | BANDAS | INDICES | MODO |
|---|---|---|---|---|---|
| 1 | `1_siamesa_6b_finetuning` | `"siamesa"` | `(0, 1, 2, 3, 4, 5)` | `False` | `"finetuning"` |
| 2 | `2_fusion_6b_finetuning` | `"fusion"` | `(0, 1, 2, 3, 4, 5)` | `False` | `"finetuning"` |
| 3 | `3_siamesa_rgb_finetuning` | `"siamesa"` | `(2, 1, 0)` | `False` | `"finetuning"` |
| 4 | `4_siamesa_6b_indices_finetuning` | `"siamesa"` | `(0, 1, 2, 3, 4, 5)` | `True` | `"finetuning"` |
| 5 | `5_siamesa_6b_cero` | `"siamesa"` | `(0, 1, 2, 3, 4, 5)` | `False` | `"cero"` |
| 6 | `6_siamesa_6b_congelado` | `"siamesa"` | `(0, 1, 2, 3, 4, 5)` | `False` | `"congelado"` |

In [ ]:
import os
import sys

try:
    import google.colab
    COLAB = True
except:
    COLAB = False

RAMA = "20-entrenamientos"
REPO = "/content/vacamuerta-metano-seg"

if COLAB:
    # si ya se clonó (por ejemplo, al correr la celda de nuevo), no se vuelve a clonar
    if not os.path.exists(REPO):
        !git clone -b {RAMA} https://github.com/sofia2005222/vacamuerta-metano-seg.git {REPO}
    %cd {REPO}
    !pip install -qq -e .
    # el notebook no ve el paquete recién instalado hasta reiniciar: agrego src/ a mano
    sys.path.insert(0, os.path.join(REPO, "src"))
else:
    %cd ..
    %load_ext autoreload
    %autoreload 2



In [ ]:
# bajar los pares del bucket
if COLAB:
    from google.colab import auth
    auth.authenticate_user()
    !mkdir -p data/pares
    !gsutil -m -q cp gs://vmseg-imagenes/pares/indice.csv "gs://vmseg-imagenes/pares/*_t1.tif" "gs://vmseg-imagenes/pares/*_t2.tif" "gs://vmseg-imagenes/pares/*_mask.tif" data/pares/


In [ ]:
import torch
from torchinfo import summary

from vmseg.config import DatasetConfig, TrainingConfig, deterministic
from vmseg.dataset_pares import get_data_pares
from vmseg.entrenar import main
from vmseg.siamesa import crear_modelo, nombres_canales

# --- experimento (tabla de docs/METODO.md) ---
NOMBRE = "1_siamesa_6b_finetuning"
ARQUITECTURA = "siamesa"      # "siamesa" o "fusion"
BANDAS = (0, 1, 2, 3, 4, 5)   # (2, 1, 0) para RGB
INDICES = False               # True agrega NDVI, NDBI y BSI
MODO = "finetuning"          # "finetuning", "congelado" o "cero"

deterministic(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

dataset_config = DatasetConfig(data_root="data/pares", bandas=BANDAS, indices=INDICES,
                               num_classes=2, batch_size=8, augment=True)
train_config = TrainingConfig(epochs=50, learning_rate=1e-3, device=device,
                              checkpoints_dir=f"runs/{NOMBRE}", pesos_clases=[1.0, 10.0])

train_loader, valid_loader, mean, std = get_data_pares(dataset_config)
print(len(train_loader.dataset), "pares de train,", len(valid_loader.dataset), "de val")


In [ ]:
canales = nombres_canales(dataset_config)
model = crear_modelo(ARQUITECTURA, canales, num_classes=dataset_config.num_classes, modo=MODO).to(device)
print(summary(model, input_size=(1, 2 * len(canales), 256, 256), depth=2))

# con el encoder congelado, solo se optimizan los parámetros que entrenan
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                              lr=train_config.learning_rate, weight_decay=train_config.weight_decay)
# baja el learning rate a la mitad si la loss de val no mejora en 5 épocas
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

history = main(model, optimizer, train_loader, valid_loader, train_config,
               num_classes=dataset_config.num_classes, scheduler=scheduler)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history["iter_train_loss"])
axes[0].set_title("Loss por iteración (train)"); axes[0].set_xlabel("Iteración")

axes[1].plot(history["train_loss"], "-o", label="train")
axes[1].plot(history["valid_loss"], "-o", label="val")
axes[1].set_title("Loss por época"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(history["train_miou"], "-o", label="train")
axes[2].plot(history["valid_miou"], "-o", label="val")
axes[2].set_title("Iou de locación"); axes[2].set_xlabel("Epoch"); axes[2].legend()

for ax in axes: ax.grid(True)
plt.show()


In [ ]:
import json
from vmseg.entrenar import guardar_curvas

mejor_epoca = int(max(range(len(history["valid_miou"])), key=lambda i: history["valid_miou"][i])) + 1
resultado = dict(nombre=NOMBRE, arquitectura=ARQUITECTURA, bandas=list(BANDAS), indices=INDICES,
                 modo=MODO, epocas=train_config.epochs, lr=train_config.learning_rate,
                 pesos_clases=train_config.pesos_clases,
                 iou_val=round(history["best_score"], 4), mejor_epoca=mejor_epoca,
                 iou_val_ult10=round(sum(history["valid_miou"][-10:]) / 10, 4),
                 pares_train=len(train_loader.dataset), pares_val=len(valid_loader.dataset))
print(resultado)

carpeta = train_config.checkpoints_dir
guardar_curvas(history, f"{carpeta}/curvas.png")
with open(f"{carpeta}/resultado.json", "w") as f:
    json.dump(resultado, f, indent=2)

if COLAB:
    destino = f"gs://vmseg-imagenes/modelos/{NOMBRE}/"
    codigo = os.system(f"gsutil cp {carpeta}/best.pt {carpeta}/curvas.png {carpeta}/resultado.json {destino}")
    if codigo == 0:
        print("subido a", destino)
    else:
        print("NO se pudo subir al bucket: bajando los archivos a la compu")
        from google.colab import files
        for f in ["best.pt", "curvas.png", "resultado.json"]:
            files.download(f"{carpeta}/{f}")